# Notebook 1 – Exploration des données
**Projet Computer Vision IG.2405 – 2026**

Ce notebook explore la structure de la base de données fournie :
- Photos de première page (présences)
- Formulaires PDF scannés
- Fichiers Excel vérité terrain

Objectif : comprendre les données avant de développer les algorithmes.

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ---------------------------------------------------------------
# CONFIG – adapter si nécessaire
# ---------------------------------------------------------------
BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
DATA_ROOT = os.path.join(BASE_DIR, 'PROJECT 2026 -DATABASE-20260518')
FORM_DIR  = os.path.join(DATA_ROOT, 'FORM1')

print('Répertoire de données :', FORM_DIR)
print('Existe :', os.path.isdir(FORM_DIR))

## 1. Inventaire des fichiers

In [ ]:
all_files = os.listdir(FORM_DIR)

imgs  = sorted([f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
pdfs  = sorted([f for f in all_files if f.lower().endswith('.pdf')])
xlsxs = sorted([f for f in all_files if f.lower().endswith('.xlsx')])

print(f'Images (photos présences) : {len(imgs)}')
print(f'Fichiers PDF (formulaires): {len(pdfs)}')
print(f'Fichiers XLSX (vérité)    : {len(xlsxs)}')

# Afficher les 5 premiers de chaque type
print('\n--- Exemples images ---')
for f in imgs[:5]: print(' ', f)
print('\n--- Exemples PDFs ---')
for f in pdfs[:5]: print(' ', f)
print('\n--- Exemples XLSX ---')
for f in xlsxs[:5]: print(' ', f)

## 2. Visualisation des photos de présence

Les étudiants prennent une photo de la **première page** du formulaire (identification + signature).

In [ ]:
# Afficher les 6 premières photos
n_show = min(6, len(imgs))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, fname in enumerate(imgs[:n_show]):
    path = os.path.join(FORM_DIR, fname)
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(fname, fontsize=8)
    axes[i].axis('off')

for j in range(n_show, len(axes)):
    axes[j].axis('off')

plt.suptitle('Photos de première page (présences)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Analyse des dimensions des images

Les photos sont prises avec des smartphones → orientations et résolutions variables.

In [ ]:
heights, widths = [], []
for fname in imgs:
    img = cv2.imread(os.path.join(FORM_DIR, fname))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)

print(f'Hauteurs : min={min(heights)}, max={max(heights)}, moy={np.mean(heights):.0f}')
print(f'Largeurs : min={min(widths)}, max={max(widths)}, moy={np.mean(widths):.0f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(heights, bins=15, color='steelblue', edgecolor='black')
ax1.set_title('Distribution des hauteurs')
ax1.set_xlabel('pixels')

ax2.hist(widths, bins=15, color='darkorange', edgecolor='black')
ax2.set_title('Distribution des largeurs')
ax2.set_xlabel('pixels')

plt.tight_layout()
plt.show()

## 4. Visualisation d'un formulaire PDF

Chaque page du formulaire PDF scanné est convertie en image pour le traitement.

In [ ]:
import fitz  # PyMuPDF

# Prendre le premier PDF disponible
sample_pdf = os.path.join(FORM_DIR, pdfs[0])
print('PDF analysé :', pdfs[0])

doc = fitz.open(sample_pdf)
print(f'Nombre de pages : {len(doc)}')

# Convertir les 4 premières pages en images
scale = 150 / 72
mat = fitz.Matrix(scale, scale)

n_pages = min(4, len(doc))
fig, axes = plt.subplots(1, n_pages, figsize=(5 * n_pages, 8))
if n_pages == 1:
    axes = [axes]

for i, page in enumerate(doc[:n_pages]):
    pix = page.get_pixmap(matrix=mat)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    axes[i].imshow(img)
    axes[i].set_title(f'Page {i+1}', fontsize=10)
    axes[i].axis('off')

doc.close()
plt.suptitle(f'Formulaire : {pdfs[0]}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Vérité terrain – fichiers XLSX

Chaque étudiant a un fichier `.xlsx` de vérité terrain contenant les onglets `PAGE-01` et `EXAM`.

In [ ]:
# Lire le premier xlsx disponible
sample_xlsx = os.path.join(FORM_DIR, xlsxs[0])
print('XLSX analysé :', xlsxs[0])

# Onglet PAGE-01
try:
    df_page1 = pd.read_excel(sample_xlsx, sheet_name='PAGE-01', header=None)
    print('\n=== PAGE-01 ===')
    print(df_page1.to_string(index=False, header=False))
except Exception as e:
    print(f'Erreur PAGE-01 : {e}')

# Onglet EXAM
try:
    df_exam = pd.read_excel(sample_xlsx, sheet_name='EXAM')
    print('\n=== EXAM (premières lignes) ===')
    print(df_exam.head(10).to_string(index=False))
except Exception as e:
    print(f'Erreur EXAM : {e}')

## 6. Correspondance images / PDFs / XLSX

La clé de correspondance est le **Student ID** dans le nom du fichier.

In [ ]:
def extract_student_id_from_filename(fname):
    """Extrait le Student ID du nom de fichier EXAM_FORM1_XXXXX.ext"""
    base = os.path.splitext(fname)[0]  # EXAM_FORM1_XXXXX
    return base.split('_')[-1]          # XXXXX

img_ids  = {extract_student_id_from_filename(f) for f in imgs}
pdf_ids  = {extract_student_id_from_filename(f) for f in pdfs}
xlsx_ids = {extract_student_id_from_filename(f) for f in xlsxs}

print(f'IDs avec photo   : {len(img_ids)}')
print(f'IDs avec PDF     : {len(pdf_ids)}')
print(f'IDs avec XLSX    : {len(xlsx_ids)}')
print(f'IDs complets (3) : {len(img_ids & pdf_ids & xlsx_ids)}')
print(f'IDs sans photo   : {pdf_ids - img_ids}')

## 7. Zoom sur la zone d'identification (page 1)

La page 1 contient les éléments clés à lire :
- Bande CODES EXAM (Module, Professeur, Date, Code)
- Grille Student ID (5 colonnes × 10 lignes)
- Grille Groupe
- Zone signature
- Cases YES/NO conditions d'examen

In [ ]:
# Prendre une image avec photo disponible
if imgs:
    sample_img = cv2.imread(os.path.join(FORM_DIR, imgs[0]))
    h, w = sample_img.shape[:2]

    # Annotation des zones principales
    annotated = sample_img.copy()
    # On normalise d'abord pour avoir des coordonnées cohérentes
    from utils.grid_decoder import normalize_page
    norm = normalize_page(sample_img)

    # Zones à annoter (coordonnées dans l'image normalisée 900×1270)
    zones = [
        ((0,   0,   900, 65),  'CODES EXAM',    (255, 100, 0)),
        ((725, 247, 155, 330), 'Student ID',     (0, 200, 0)),
        ((516, 247, 105, 330), 'Groupe',          (200, 0, 200)),
        ((30,  272, 372, 288), 'Signature',       (0, 0, 255)),
    ]

    vis = norm.copy()
    for (x, y, w_r, h_r), label, color in zones:
        cv2.rectangle(vis, (x, y), (x + w_r, y + h_r), color, 3)
        cv2.putText(vis, label, (x + 5, y + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    plt.figure(figsize=(8, 11))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title('Page 1 normalisée – zones principales annotées', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Bilan

| Élément | Constat |
|---|---|
| Photos présences | Résolutions variables, nécessitent deskew |
| PDFs scannés | Plusieurs pages, qualité uniforme |
| Vérités terrain | 2 onglets (PAGE-01, EXAM) par étudiant |
| Structure | Student ID dans le nom de fichier = clé de liaison |

**Prochaine étape** → `02_preprocessing_alignement.ipynb` : correction d'inclinaison et normalisation.